# <span style="color:#16A34A">Tuning the Baseline Until It Generalizes</span>

<table width="100%" cellpadding="10" cellspacing="0">
<tr bgcolor="#f8fafc"><td>
<p><b>Fundamentals of Natural Language Processing</b> | Universitat Autonoma de Barcelona | 2025 2026</p>
<p>Phoebe Iglesias (1713459), David Redrejo (1790336), Pau Rossell (1750424)</p>
<h3><font color="#16A34A">Notebook question</font></h3>
<p>Which changes improve validation accuracy, and which changes only memorize the training set?</p>
<h3><font color="#16A34A">Connection with the previous notebook</font></h3>
<p>This follows <b><font color="#2563EB">Notebook 02: Building the First Classical Baseline</font></b>. The first baseline worked, but it also gave us choices to challenge.</p>
<h3><font color="#16A34A">What this chapter contributes</font></h3>
<p>We compare controlled model variants, choose the best validated SVM, and build the recommended submission.</p>
</td></tr>
</table>

<table width="100%" cellpadding="8" cellspacing="0">
<tr bgcolor="#16A34A"><th><font color="white">Move</font></th><th><font color="white">What we try to understand</font></th></tr>
<tr><td>1</td><td>Diagnose</td></tr><tr><td>2</td><td>Load</td></tr><tr><td>3</td><td>Collapse labels</td></tr><tr><td>4</td><td>Inspect imbalance</td></tr><tr><td>5</td><td>Compare models</td></tr><tr><td>6</td><td>Read gap</td></tr><tr><td>7</td><td>Analyze errors</td></tr><tr><td>8</td><td>Submit</td></tr>
</table>

## Chapter Map

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

chapter_color = "#16A34A"
labels = ['Diagnose', 'Load', 'Collapse labels', 'Inspect imbalance', 'Compare models', 'Read gap', 'Analyze errors', 'Submit']

fig, ax = plt.subplots(figsize=(14, 2.6))
ax.set_xlim(0, len(labels))
ax.set_ylim(0, 1)
ax.axis("off")

for idx, label in enumerate(labels):
    card = FancyBboxPatch(
        (idx + 0.06, 0.25), 0.88, 0.48,
        boxstyle="round,pad=0.04,rounding_size=0.05",
        linewidth=1.4,
        edgecolor=chapter_color,
        facecolor="#f8fafc"
    )
    ax.add_patch(card)
    ax.text(idx + 0.5, 0.49, label, ha="center", va="center", fontsize=10.5, color="#0f172a", wrap=True)
    if idx < len(labels) - 1:
        ax.annotate("", xy=(idx + 1.02, 0.49), xytext=(idx + 0.94, 0.49), arrowprops=dict(arrowstyle=">", color=chapter_color, lw=1.8))

ax.text(0.02, 0.9, "How this notebook moves", fontsize=14, weight="bold", color=chapter_color)
plt.show()

## Previous Baseline Weaknesses

We begin by being honest about the first baseline. It was useful, but its settings were not sacred. Balanced weights, rare feature filtering, and longer character grams all deserve to be tested against the actual metric.

## External Evidence

The literature supports the same instinct we got from the data: interpretable lexical models can remain competitive for ICD coding, especially when the text is short and labels are imbalanced. We use that as guidance, not as a script.

## Improvement Plan

The plan is to compare models on the same split, read both training and validation accuracy, and avoid choosing a model just because it memorizes more. The gap between train and validation becomes part of the evidence.

In [ ]:
import os
import sys
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.svm import LinearSVC

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "src").exists() else NOTEBOOK_DIR.parent
DATA_DIR = PROJECT_ROOT / "data"
SUBMISSION_DIR = PROJECT_ROOT / "submissions"
SUBMISSION_DIR.mkdir(exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from data_processing import normalize_texts, extract_category
from evaluation import generate_submission

required_files = [
    DATA_DIR / "codification_data.csv",
    DATA_DIR / "leaderboard_data.csv",
    DATA_DIR / "icd_d_p_pairs.csv",
]
missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing required data files. Place the project CSV files in the data/ folder: "
        + ", ".join(missing)
    )

print(f"Project root : {PROJECT_ROOT}")
print(f"Data folder  : {DATA_DIR}")
print(f"Submissions  : {SUBMISSION_DIR}")
print("Setup complete.")

## Data Loading

We reload the same three resources so the improved notebook is self contained. The ICD catalog is also loaded because we want to test whether official descriptions can add useful language later.

In [ ]:
codif_df = pd.read_csv(DATA_DIR / 'codification_data.csv')
lead_df = pd.read_csv(DATA_DIR / 'leaderboard_data.csv')
icd_df = pd.read_csv(DATA_DIR / 'icd_d_p_pairs.csv')

print(f'Training rows:   {len(codif_df):,}')
print(f'Unique literals: {codif_df["Literal"].nunique():,}')
print(f'Unique codes:    {codif_df["Code"].nunique():,}')
print(f'Leaderboard:     {len(lead_df):,}')
print(f'ICD descriptions:{len(icd_df):,}')

display(codif_df.head())
display(lead_df.head())
display(icd_df.head())

## One Category Per Literal

We rebuild the single label dataset using majority vote per literal. This keeps the experiment comparable to the baseline while still counting how much ambiguity existed before simplification.

In [ ]:
cat_df = codif_df.copy()
cat_df['y_category'] = cat_df['Code'].apply(extract_category)

ambiguity = (
    cat_df.groupby('Literal')['y_category']
    .nunique()
    .reset_index(name='n_categories')
)
ambiguous_literals = ambiguity[ambiguity['n_categories'] > 1]

cat_df = (
    cat_df.groupby('Literal')['y_category']
    .agg(lambda s: s.value_counts().index[0])
    .reset_index()
)
cat_df['text_norm'] = normalize_texts(cat_df['Literal'])

print(f'Rows after one-label-per-literal: {len(cat_df):,}')
print(f'Categories: {cat_df["y_category"].nunique()}')
print(f'Ambiguous original literals: {len(ambiguous_literals):,}')

display(cat_df.head(10))

## Label Imbalance

Before comparing models, we look again at the category distribution. This matters because strict accuracy rewards common categories more strongly than rare ones.

In [ ]:
label_counts = cat_df['y_category'].value_counts().sort_values(ascending=False)

plt.figure(figsize=(13, 4))
plt.bar(label_counts.index.astype(str), label_counts.values, color='#2f6f6d')
plt.title('ICD category distribution after majority vote')
plt.xlabel('Category')
plt.ylabel('Number of unique literals')
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.25)
plt.show()

print(label_counts.head(10))

## Train Validation Split

We keep a stratified split so every model faces the same validation problem. Without this, the comparison would mix model quality with split luck.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    cat_df['text_norm'],
    cat_df['y_category'],
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=cat_df['y_category'],
)

print(f'Train size: {len(X_train):,}')
print(f'Val size:   {len(X_val):,}')

## Model Comparison

Now we try four variants. One recreates the baseline, one is tuned for shorter fragments, one adds capacity, and one mixes word and character features. We are not just asking which one trains best. We are asking which one travels best to validation.

In [ ]:
def evaluate_model(name, vectorizer, classifier, X_train, y_train, X_val, y_val):
    start = time.time()
    X_train_vec = vectorizer.fit_transform(X_train)
    X_val_vec = vectorizer.transform(X_val)
    classifier.fit(X_train_vec, y_train)

    train_pred = classifier.predict(X_train_vec)
    val_pred = classifier.predict(X_val_vec)

    result = {
        'model': name,
        'features': X_train_vec.shape[1],
        'train_accuracy': accuracy_score(y_train, train_pred),
        'val_accuracy': accuracy_score(y_val, val_pred),
        'val_weighted_f1': f1_score(y_val, val_pred, average='weighted', zero_division=0),
        'val_macro_f1': f1_score(y_val, val_pred, average='macro', zero_division=0),
        'seconds': time.time() - start,
    }
    return result, vectorizer, classifier, val_pred

model_specs = [
    (
        '02 baseline: char(3,6), balanced',
        TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 6), sublinear_tf=True,
                        max_features=100_000, min_df=2, dtype=np.float32),
        LinearSVC(C=1.0, class_weight='balanced', max_iter=10_000, random_state=RANDOM_STATE),
    ),
    (
        'Improved: char(2,5), C=2',
        TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), sublinear_tf=True,
                        max_features=200_000, min_df=1, dtype=np.float32),
        LinearSVC(C=2.0, class_weight=None, max_iter=10_000, random_state=RANDOM_STATE),
    ),
    (
        'Higher capacity: char(2,6), C=5',
        TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 6), sublinear_tf=True,
                        max_features=300_000, min_df=1, dtype=np.float32),
        LinearSVC(C=5.0, class_weight=None, max_iter=10_000, random_state=RANDOM_STATE),
    ),
    (
        'Word + char high training accuracy',
        FeatureUnion([
            ('word', TfidfVectorizer(analyzer='word', ngram_range=(1, 3), sublinear_tf=True,
                                     min_df=1, max_features=150_000, dtype=np.float32)),
            ('char', TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 6), sublinear_tf=True,
                                     min_df=1, max_features=300_000, dtype=np.float32)),
        ]),
        LinearSVC(C=5.0, class_weight=None, max_iter=10_000, random_state=RANDOM_STATE),
    ),
]

results = []
fitted = {}
for name, vec, clf in model_specs:
    result, vec_fit, clf_fit, val_pred = evaluate_model(name, vec, clf, X_train, y_train, X_val, y_val)
    results.append(result)
    fitted[name] = (vec_fit, clf_fit, val_pred)
    print(f'{name}: train={result["train_accuracy"]:.4f}, val={result["val_accuracy"]:.4f}')

results_df = pd.DataFrame(results).sort_values('val_accuracy', ascending=False)
display(results_df)

## Model Comparison Reading

The healthiest model is the character configuration with shorter grams and no class weights. It improves validation while the word plus character model mostly improves training. That difference is the overfitting warning we were looking for.

In [ ]:
plot_df = results_df.sort_values('train_accuracy')

plt.figure(figsize=(10, 5))
y = np.arange(len(plot_df))
plt.barh(y - 0.18, plot_df['train_accuracy'], height=0.35, label='Train accuracy', color='#5271a3')
plt.barh(y + 0.18, plot_df['val_accuracy'], height=0.35, label='Validation accuracy', color='#d9822b')
plt.yticks(y, plot_df['model'])
plt.xlim(0.50, 1.00)
plt.xlabel('Accuracy')
plt.title('Training accuracy vs validation accuracy')
plt.grid(axis='x', alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

## ICD Description Augmentation

We also leave an optional experiment with official descriptions. It is promising because the catalog adds medical language, but it changes the training distribution, so we treat it as an experiment rather than the final default.

In [ ]:
RUN_ICD_AUGMENTATION = False

if RUN_ICD_AUGMENTATION:
    icd_aug = icd_df.copy()
    icd_aug['y_category'] = icd_aug['Code'].apply(extract_category)
    icd_aug['text_norm'] = normalize_texts(icd_aug['Description'])
    icd_aug = icd_aug[
        icd_aug['y_category'].isin(set(y_train)) &
        (icd_aug['text_norm'].str.len() > 0)
    ]

    X_aug = list(X_train) * 2 + list(icd_aug['text_norm'])
    y_aug = list(y_train) * 2 + list(icd_aug['y_category'])

    aug_vec = TfidfVectorizer(
        analyzer='char_wb', ngram_range=(2, 5), sublinear_tf=True,
        max_features=200_000, min_df=1, dtype=np.float32
    )
    aug_clf = LinearSVC(C=2.0, class_weight=None, max_iter=10_000, random_state=RANDOM_STATE)

    aug_result, aug_vec, aug_clf, aug_val_pred = evaluate_model(
        'ICD-description augmented model', aug_vec, aug_clf, X_aug, y_aug, X_val, y_val
    )

    original_train_pred = aug_clf.predict(aug_vec.transform(X_train))
    aug_result['train_accuracy_on_original_literals'] = accuracy_score(y_train, original_train_pred)
    display(pd.DataFrame([aug_result]))
else:
    print('Skipped. Set RUN_ICD_AUGMENTATION = True to run this slower experiment.')

## Confusion Matrix Reading

The confusion matrix shows where categories collide. This is where we stop thinking only in one accuracy number and inspect the shape of the mistakes.

In [ ]:
selected_name = 'Improved: char(2,5), C=2'
selected_vec, selected_clf, selected_val_pred = fitted[selected_name]

labels = sorted(cat_df['y_category'].unique())
cm = confusion_matrix(y_val, selected_val_pred, labels=labels)
cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)

plt.figure(figsize=(12, 10))
plt.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
plt.title('Normalized confusion matrix: selected improved model')
plt.xlabel('Predicted category')
plt.ylabel('True category')
plt.xticks(range(len(labels)), labels, fontsize=8)
plt.yticks(range(len(labels)), labels, fontsize=8)
plt.colorbar(label='Row-normalized proportion')
plt.tight_layout()
plt.show()

print(classification_report(y_val, selected_val_pred, zero_division=0))

## Feature Weight Reading

A linear SVM lets us look inside the model. The top weighted character fragments show whether the classifier learned medical roots and abbreviations or just noise.

In [ ]:
def show_top_features_for_class(vectorizer, classifier, class_label, top_n=15):
    feature_names = np.array(vectorizer.get_feature_names_out())
    class_index = list(classifier.classes_).index(class_label)
    weights = classifier.coef_[class_index]
    top_idx = np.argsort(weights)[-top_n:][::-1]
    return pd.DataFrame({
        'category': class_label,
        'feature': feature_names[top_idx],
        'weight': weights[top_idx],
    })

for class_label in ['O', 'Z', 'J', 'I', 'N']:
    if class_label in selected_clf.classes_:
        display(show_top_features_for_class(selected_vec, selected_clf, class_label, top_n=12))

## Final Submission

After choosing the best validated setting, we retrain on all available unique literals and generate the final CSV. The checks at the end make sure the output is submission ready.

In [ ]:
final_vec = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(2, 5),
    sublinear_tf=True,
    max_features=200_000,
    min_df=1,
    dtype=np.float32,
)
final_clf = LinearSVC(
    C=2.0,
    class_weight=None,
    max_iter=10_000,
    random_state=RANDOM_STATE,
)

X_all = cat_df['text_norm']
y_all = cat_df['y_category']

X_all_vec = final_vec.fit_transform(X_all)
final_clf.fit(X_all_vec, y_all)

train_pred_all = final_clf.predict(X_all_vec)
final_train_accuracy = accuracy_score(y_all, train_pred_all)

print(f'Final training accuracy on all unique literals: {final_train_accuracy:.4f}')
print(f'Feature matrix: {X_all_vec.shape}')

In [ ]:
lead_norm = normalize_texts(lead_df["Literal"])
X_lead = final_vec.transform(lead_norm)
y_lead_pred = final_clf.predict(X_lead)

submission_path = SUBMISSION_DIR / "svm_improved_training_accuracy.csv"
submission_df = generate_submission(
    lead_df,
    y_lead_pred,
    output_path=str(submission_path),
)

expected_columns = ["id", "Literal", "y_category"]
assert list(submission_df.columns) == expected_columns
assert len(submission_df) == len(lead_df)
assert submission_df["y_category"].notna().all()
assert (submission_df["y_category"].astype(str).str.len() > 0).all()
assert submission_path.exists()

print("Submission preview:")
display(submission_df.head(10))
print()
print("Category distribution in submission:")
print(submission_df["y_category"].value_counts().sort_index())

## Improved Model Takeaways

This notebook gives the strongest classical model in the project. It becomes the target that <b><font color="#7C3AED">Notebook 04: Testing a GPU Transformer Against the Classical Model</font></b> must beat with a GPU transformer.